In [ ]:
from google.colab import files
uploaded = files.upload()

Saving gold_sample_300_master_clean.csv to gold_sample_300_master_clean.csv
Saving gold_standard_200_events.csv to gold_standard_200_events.csv


## **Step 1:** Standardize and Merge Data Files

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Standardize the gold_standard_200_events.csv file
def standardize_gold_standard_200(df):

    print("Standardizing gold_standard_200_events.csv...")

    # Create standardized dataframe with consistent column names
    standardized = pd.DataFrame()

    # Core columns
    standardized['text'] = df['raw_description']
    standardized['event'] = df['verified_event_name']
    standardized['venue'] = df['verified_venue']
    standardized['datetime'] = df['verified_start_datetime']

    # Metadata columns
    standardized['city'] = df['city']
    standardized['source'] = df['source']
    standardized['url'] = df['url']
    standardized['quality_score'] = df['quality_score']
    standardized['annotation_difficulty'] = df['annotation_difficulty']
    standardized['annotation_complete'] = df['annotation_complete']
    standardized['data_source'] = 'gold_standard_200'

    # Clean up
    standardized = standardized.replace('nan', np.nan)
    standardized = standardized.replace('', np.nan)

    return standardized

# Standardize the gold_sample_300_master_clean.csv file
def standardize_gold_sample_987(df):

    print("Standardizing gold_sample_300_master_clean.csv...")

    # Create standardized dataframe
    standardized = pd.DataFrame()

    # Core columns
    standardized['text'] = df['raw_description']
    standardized['event'] = df['label_event_name']
    standardized['venue'] = df['label_venue']
    standardized['datetime'] = df['label_start_datetime_iso']

    # Metadata columns
    standardized['city'] = df['city']
    standardized['source'] = df['source']
    standardized['url'] = df['url']
    standardized['quality_score'] = np.nan  # Not available in this dataset
    standardized['annotation_difficulty'] = np.nan  # Not available
    standardized['annotation_complete'] = True  # Assume complete since in dataset
    standardized['data_source'] = 'gold_sample_987'

    # Clean up
    standardized = standardized.replace('nan', np.nan)
    standardized = standardized.replace('', np.nan)

    return standardized

# Create master dataset from both gold standard files
def create_master_dataset():


    print("="*60)
    print("CREATING MASTER TRAINING DATASET")
    print("="*60)

    # Load both files
    print("\n1. Loading data files...")
    gold_200 = pd.read_csv('gold_standard_200_events.csv')
    print(f"Loaded {len(gold_200)} events from gold_standard_200_events.csv")

    gold_987 = pd.read_csv('gold_sample_300_master_clean.csv')
    print(f"Loaded {len(gold_987)} events from gold_sample_300_master_clean.csv")

    # Standardize both datasets
    print("\n2. Standardizing column names...")
    standardized_200 = standardize_gold_standard_200(gold_200)
    standardized_987 = standardize_gold_sample_987(gold_987)

    # Combine datasets
    print("\n3. Combining datasets...")
    master_df = pd.concat([standardized_200, standardized_987], ignore_index=True)
    print(f"Combined total: {len(master_df)} events")

    # Remove duplicates based on text
    print("\n4. Removing duplicates...")
    initial_count = len(master_df)
    master_df = master_df.drop_duplicates(subset=['text'], keep='first')
    removed_count = initial_count - len(master_df)
    print(f"Removed {removed_count} duplicate texts")
    print(f"Final count: {len(master_df)} unique events")

    # Data quality check
    print("\n5. Data quality check...")

    # Check for missing values in core columns
    missing_text = master_df['text'].isna().sum()
    missing_event = master_df['event'].isna().sum()
    missing_venue = master_df['venue'].isna().sum()
    missing_datetime = master_df['datetime'].isna().sum()

    print(f"Missing text: {missing_text}")
    print(f"Missing event names: {missing_event}")
    print(f"Missing venues: {missing_venue}")
    print(f"Missing datetimes: {missing_datetime}")

    # Remove rows with missing text (can't train without text)
    master_df = master_df.dropna(subset=['text'])

    # Filter to rows with at least one annotation
    has_annotation = (
        master_df['event'].notna() |
        master_df['venue'].notna() |
        master_df['datetime'].notna()
    )
    master_df = master_df[has_annotation]

    print(f"\n6. Final dataset: {len(master_df)} events with text and at least one annotation")

    # Distribution by source
    print("\n7. Dataset composition:")
    source_counts = master_df['data_source'].value_counts()
    for source, count in source_counts.items():
        percentage = (count / len(master_df)) * 100
        print(f"   - {source}: {count} events ({percentage:.1f}%)")

    # Save master file
    output_file = 'master_training_data.csv'
    master_df.to_csv(output_file, index=False)
    print(f"\n8. Master dataset saved to: {output_file}")

    # Create sample for verification
    print("\n9. Sample of master dataset:")
    print("-"*60)
    for i, row in master_df.head(3).iterrows():
        print(f"\nExample {i+1}:")
        print(f"Text: {row['text'][:100]}...")
        print(f"Event: {row['event']}")
        print(f"Venue: {row['venue']}")
        print(f"DateTime: {row['datetime']}")
        print(f"Source: {row['data_source']}")

    # Statistics summary
    print("\n" + "="*60)
    print("MASTER DATASET STATISTICS")
    print("="*60)
    print(f"Total events: {len(master_df)}")
    print(f"Events with event name: {master_df['event'].notna().sum()}")
    print(f"Events with venue: {master_df['venue'].notna().sum()}")
    print(f"Events with datetime: {master_df['datetime'].notna().sum()}")

    # All three annotations
    all_three = (
        master_df['event'].notna() &
        master_df['venue'].notna() &
        master_df['datetime'].notna()
    )
    print(f"Events with all three annotations: {all_three.sum()}")

    return master_df

if __name__ == "__main__":
    master_df = create_master_dataset()
    print("\n  Master dataset creation complete!")

CREATING MASTER TRAINING DATASET

1. Loading data files...
Loaded 200 events from gold_standard_200_events.csv
Loaded 987 events from gold_sample_300_master_clean.csv

2. Standardizing column names...
Standardizing gold_standard_200_events.csv...
Standardizing gold_sample_300_master_clean.csv...

3. Combining datasets...
Combined total: 1187 events

4. Removing duplicates...
Removed 682 duplicate texts
Final count: 505 unique events

5. Data quality check...
Missing text: 0
Missing event names: 0
Missing venues: 193
Missing datetimes: 346

6. Final dataset: 505 events with text and at least one annotation

7. Dataset composition:
   - gold_sample_987: 346 events (68.5%)
   - gold_standard_200: 159 events (31.5%)

8. Master dataset saved to: master_training_data.csv

9. Sample of master dataset:
------------------------------------------------------------

Example 1:
Text: Action Navigation

Saturday, May 18, 2024 - 7:30 PM to 11:59 PM (Click on the image to go to Q2 Stad...
Event: Majo

## **Step 2:** Create Train/Test Split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Create train/test split from master datatset
def create_train_test_split(test_size=0.15, random_state=42):

    print("="*60)
    print("Creating Train/Test Split")
    print("="*60)

    # Load master dataset
    print("\n1. Loading master dataset...")
    master_df = pd.read_csv('master_training_data.csv')
    print(f"   - Loaded {len(master_df)} events")

    # Stratify by data source if possible
    print("\n2. Creating split...")
    print(f"   - Test size: {test_size*100:.0f}%")
    print(f"   - Random state: {random_state}")

    # Try stratify by source to maintain distribution
    try:
        train_df, test_df = train_test_split(
            master_df,
            test_size=test_size,
            random_state=random_state,
            stratify=master_df['data_source']
        )
        print("   - Using stratified split by data source")
    except:
        # If stratification fails, do regular split
        train_df, test_df = train_test_split(
            master_df,
            test_size=test_size,
            random_state=random_state
        )
        print("   - Using regular random split")

    print(f"\n3. Split results:")
    print(f"   - Training set: {len(train_df)} events ({len(train_df)/len(master_df)*100:.1f}%)")
    print(f"   - Test set: {len(test_df)} events ({len(test_df)/len(master_df)*100:.1f}%)")

    # Check distribution in each set
    print("\n4. Training set composition:")
    train_sources = train_df['data_source'].value_counts()
    for source, count in train_sources.items():
        percentage = (count / len(train_df)) * 100
        print(f"   - {source}: {count} events ({percentage:.1f}%)")

    print("\n5. Test set composition:")
    test_sources = test_df['data_source'].value_counts()
    for source, count in test_sources.items():
        percentage = (count / len(test_df)) * 100
        print(f"   - {source}: {count} events ({percentage:.1f}%)")

    # Check annotation completeness in each set
    print("\n6. Training set annotation coverage:")
    train_event = train_df['event'].notna().sum()
    train_venue = train_df['venue'].notna().sum()
    train_datetime = train_df['datetime'].notna().sum()
    print(f"   - Events with event name: {train_event} ({train_event/len(train_df)*100:.1f}%)")
    print(f"   - Events with venue: {train_venue} ({train_venue/len(train_df)*100:.1f}%)")
    print(f"   - Events with datetime: {train_datetime} ({train_datetime/len(train_df)*100:.1f}%)")

    print("\n7. Test set annotation coverage:")
    test_event = test_df['event'].notna().sum()
    test_venue = test_df['venue'].notna().sum()
    test_datetime = test_df['datetime'].notna().sum()
    print(f"   - Events with event name: {test_event} ({test_event/len(test_df)*100:.1f}%)")
    print(f"   - Events with venue: {test_venue} ({test_venue/len(test_df)*100:.1f}%)")
    print(f"   - Events with datetime: {test_datetime} ({test_datetime/len(test_df)*100:.1f}%)")

    # Save train and test subsets
    train_file = 'train_data.csv'
    test_file = 'test_data.csv'

    train_df.to_csv(train_file, index=False)
    test_df.to_csv(test_file, index=False)

    print(f"\n8. Files saved:")
    print(f"   - Training data: {train_file}")
    print(f"   - Test data: {test_file}")

    # Show examples from each subset
    print("\n9. Sample from training set:")
    print("-"*60)
    for i, row in train_df.head(2).iterrows():
        print(f"\nTraining Example {i+1}:")
        print(f"Text: {row['text'][:80]}...")
        print(f"Event: {row['event']}")
        print(f"Venue: {row['venue']}")

    print("\n10. Sample from test set:")
    print("-"*60)
    for i, row in test_df.head(2).iterrows():
        print(f"\nTest Example {i+1}:")
        print(f"Text: {row['text'][:80]}...")
        print(f"Event: {row['event']}")
        print(f"Venue: {row['venue']}")

    return train_df, test_df

# Verify there's no overlap between train and test sets
def verify_no_overlap():

    print("\n" + "="*60)
    print("VERIFYING TRAIN/TEST SPLIT INTEGRITY")
    print("="*60)

    train_df = pd.read_csv('train_data.csv')
    test_df = pd.read_csv('test_data.csv')

    # Check for text overlap
    train_texts = set(train_df['text'].dropna())
    test_texts = set(test_df['text'].dropna())
    overlap = train_texts.intersection(test_texts)

    if len(overlap) == 0:
        print("✅ No overlap found between train and test sets")
    else:
        print(f"⚠️ Warning: {len(overlap)} texts appear in both train and test sets")

    # Summary statistics
    total = len(train_df) + len(test_df)
    print(f"\nTotal events: {total}")
    print(f"Train: {len(train_df)} ({len(train_df)/total*100:.1f}%)")
    print(f"Test: {len(test_df)} ({len(test_df)/total*100:.1f}%)")

    return len(overlap) == 0

if __name__ == "__main__":
    # Create the split
    train_df, test_df = create_train_test_split(test_size=0.15, random_state=42)

    # Verify integrity
    is_valid = verify_no_overlap()

    if is_valid:
        print("\n✅ Train/test split creation complete!")
    else:
        print("\n⚠️ Train/test split complete but with warnings")

Creating Train/Test Split

1. Loading master dataset...
   - Loaded 505 events

2. Creating split...
   - Test size: 15%
   - Random state: 42
   - Using stratified split by data source

3. Split results:
   - Training set: 429 events (85.0%)
   - Test set: 76 events (15.0%)

4. Training set composition:
   - gold_sample_987: 294 events (68.5%)
   - gold_standard_200: 135 events (31.5%)

5. Test set composition:
   - gold_sample_987: 52 events (68.4%)
   - gold_standard_200: 24 events (31.6%)

6. Training set annotation coverage:
   - Events with event name: 429 (100.0%)
   - Events with venue: 264 (61.5%)
   - Events with datetime: 135 (31.5%)

7. Test set annotation coverage:
   - Events with event name: 76 (100.0%)
   - Events with venue: 48 (63.2%)
   - Events with datetime: 24 (31.6%)

8. Files saved:
   - Training data: train_data.csv
   - Test data: test_data.csv

9. Sample from training set:
------------------------------------------------------------

Training Example 331:
Tex

## **Step 3:** Prepare Training Data for spaCy

In [ ]:

import re
import json
from typing import List, Tuple, Dict

# Find all occurrences of entity in text (case-not-sensitive)
def find_entity_in_text(text: str, entity: str) -> List[Tuple[int, int]]:

    positions = []
    if not entity or pd.isna(entity) or entity == 'nan':
        return positions

    entity = str(entity).strip()
    text_lower = text.lower()
    entity_lower = entity.lower()

    # Try exact match
    start = 0
    while True:
        pos = text_lower.find(entity_lower, start)
        if pos == -1:
            break
        positions.append((pos, pos + len(entity)))
        start = pos + 1

    # If no exact match, try partial match (first 3 words)
    if not positions and len(entity.split()) > 3:
        entity_words = entity.split()[:3]
        partial_entity = ' '.join(entity_words)
        partial_lower = partial_entity.lower()

        start = 0
        while True:
            pos = text_lower.find(partial_lower, start)
            if pos == -1:
                break
            positions.append((pos, pos + len(partial_entity)))
            start = pos + 1

    return positions

# Very important (since missing during collection) - find datetime patterns in text
def find_datetime_in_text(text: str) -> List[Tuple[int, int]]:

    positions = []

    # Common date patterns
    patterns = [
        # Month day, year
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2}(?:st|nd|rd|th)?(?:,? \d{4})?\b',
        # MM/DD/YYYY or MM/DD
        r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b',
        # Day of week + date
        r'\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)[,\s]+[A-Z][a-z]+ \d{1,2}\b',
        # Time
        r'\b\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)\b',
        # Relative dates
        r'\b(?:Tonight|Tomorrow|Today)\b',
        # Date ranges
        r'\b\d{1,2}(?:st|nd|rd|th)[-\s]+\d{1,2}(?:st|nd|rd|th)\b',
        # Short day + date
        r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)[,\s]+\d{1,2}/\d{1,2}\b',
    ]

    for pattern in patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE)
        for match in matches:
            positions.append((match.start(), match.end()))

    # Remove overlapping positions
    positions = sorted(positions, key=lambda x: x[0])
    cleaned_positions = []
    last_end = -1
    for start, end in positions:
        if start >= last_end:
            cleaned_positions.append((start, end))
            last_end = end

    return cleaned_positions

# Convert dataframe to spaCy training format
def create_spacy_training_data(df: pd.DataFrame) -> List[Tuple[str, Dict]]:

    training_data = []
    stats = {
        'total': len(df),
        'with_entities': 0,
        'no_entities': 0,
        'event_found': 0,
        'venue_found': 0,
        'datetime_found': 0
    }

    print(f"Processing {len(df)} annotations...")

    for idx, row in df.iterrows():
        if pd.isna(row.get('text')):
            continue

        text = str(row['text'])
        entities = []

        # Find EVENT entities
        if pd.notna(row.get('event')) and str(row.get('event')) != 'nan':
            event_positions = find_entity_in_text(text, row['event'])
            if event_positions:
                for start, end in event_positions[:1]:  # Take first occurrence
                    entities.append((start, end, 'EVENT'))
                stats['event_found'] += 1

        # Find VENUE entities
        if pd.notna(row.get('venue')) and str(row.get('venue')) != 'nan':
            venue_positions = find_entity_in_text(text, row['venue'])
            if venue_positions:
                for start, end in venue_positions[:1]:  # Take first occurrence
                    entities.append((start, end, 'VENUE'))
                stats['venue_found'] += 1

        # Find DATETIME entities
        datetime_positions = find_datetime_in_text(text)
        if datetime_positions:
            for start, end in datetime_positions[:1]:  # Take first occurrence
                entities.append((start, end, 'DATETIME'))
            stats['datetime_found'] += 1

        # Sort and clean entities
        entities = sorted(entities, key=lambda x: x[0])

        # Remove overlapping entities
        cleaned_entities = []
        last_end = -1
        for start, end, label in entities:
            if start >= last_end:
                cleaned_entities.append((start, end, label))
                last_end = end

        if cleaned_entities:
            training_data.append((text, {"entities": cleaned_entities}))
            stats['with_entities'] += 1
        else:
            stats['no_entities'] += 1

        # Progress update
        if (idx + 1) % 100 == 0:
            print(f"   Processed {idx + 1}/{len(df)} annotations...")

    return training_data, stats

# Prepare training data from train_data.csv
def prepare_all_training_data():


    print("="*60)
    print("PREPARING TRAINING DATA FOR SPACY")
    print("="*60)

    # Load training data
    print("\n1. Loading training data...")
    train_df = pd.read_csv('train_data.csv')
    print(f"   - Loaded {len(train_df)} training examples")

    # Create spaCy training data
    print("\n2. Converting to spaCy format...")
    training_data, stats = create_spacy_training_data(train_df)

    # Display statistics
    print("\n3. Training data statistics:")
    print(f"   - Total texts: {stats['total']}")
    print(f"   - Texts with entities: {stats['with_entities']}")
    print(f"   - Texts without entities: {stats['no_entities']}")
    print(f"   - Events found in text: {stats['event_found']}")
    print(f"   - Venues found in text: {stats['venue_found']}")
    print(f"   - DateTimes found in text: {stats['datetime_found']}")

    # Save training data
    print("\n4. Saving training data...")

    # Save as JSON for easy loading
    with open('spacy_training_data.json', 'w') as f:
        # Convert to JSON-serializable format
        json_data = []
        for text, annots in training_data:
            json_data.append({
                'text': text,
                'entities': annots['entities']
            })
        json.dump(json_data, f, indent=2)

    print(f"   - Saved {len(training_data)} training examples to spacy_training_data.json")

    # Display samples
    print("\n5. Sample training data:")
    print("-"*60)
    for i, (text, annots) in enumerate(training_data[:3]):
        print(f"\nExample {i+1}:")
        print(f"Text: {text[:100]}...")
        print(f"Entities:")
        for start, end, label in annots['entities']:
            entity_text = text[start:end]
            print(f"   - {label}: '{entity_text}' at position ({start}, {end})")

    return training_data

# Verify quality of prepared training data
def verify_training_data():

    print("\n" + "="*60)
    print("VERIFYING TRAINING DATA QUALITY")
    print("="*60)

    with open('spacy_training_data.json', 'r') as f:
        data = json.load(f)

    # Check for various issues
    issues = {
        'overlapping_entities': 0,
        'empty_entities': 0,
        'out_of_bounds': 0
    }

    entity_counts = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}

    for item in data:
        text = item['text']
        entities = item['entities']

        # Check for overlaps
        for i in range(len(entities) - 1):
            if entities[i][1] > entities[i+1][0]:
                issues['overlapping_entities'] += 1

        # Check for empty or out of bounds
        for start, end, label in entities:
            if start >= end:
                issues['empty_entities'] += 1
            if start < 0 or end > len(text):
                issues['out_of_bounds'] += 1

            entity_counts[label] = entity_counts.get(label, 0) + 1

    print(f"\nTotal training examples: {len(data)}")
    print(f"\nEntity distribution:")
    for label, count in entity_counts.items():
        print(f"   - {label}: {count}")

    print(f"\nQuality issues found:")
    for issue, count in issues.items():
        if count > 0:
            print(f"   ⚠️ {issue}: {count}")

    if sum(issues.values()) == 0:
        print("   ✅ No quality issues found!")

    return sum(issues.values()) == 0

if __name__ == "__main__":
    # Prepare training data
    training_data = prepare_all_training_data()

    # Verify quality
    is_valid = verify_training_data()

    if is_valid:
        print("\n✅ Training data preparation complete!")
    else:
        print("\n⚠️ Training data prepared with some warnings")

PREPARING TRAINING DATA FOR SPACY

1. Loading training data...
   - Loaded 429 training examples

2. Converting to spaCy format...
Processing 429 annotations...
   Processed 100/429 annotations...
   Processed 200/429 annotations...
   Processed 300/429 annotations...
   Processed 400/429 annotations...

3. Training data statistics:
   - Total texts: 429
   - Texts with entities: 418
   - Texts without entities: 11
   - Events found in text: 349
   - Venues found in text: 198
   - DateTimes found in text: 165

4. Saving training data...
   - Saved 418 training examples to spacy_training_data.json

5. Sample training data:
------------------------------------------------------------

Example 1:
Text: Denver Symphony Pops Concert - Movie Music Spectacular. Orchestra performing popular film soundtrack...
Entities:
   - EVENT: 'Denver Symphony Pops Concert' at position (0, 28)

Example 2:
Text: Austin Country Music Festival at Leander- Cedar Park, April 6, 2024 - Saturday, April 6, 2024 at

## **Step 4:** Train Transformer NER Model

In [ ]:
!pip install spacy-lookups-data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 9.5 MB/s eta 0:00:00


In [ ]:
import os
# Use force kill command
os.kill(os.getpid(), 9)

In [ ]:
#!pip install spacy-lookups-data

import spacy
import pandas as pd
import json
from spacy.training import Example
from spacy.util import minibatch, compounding
from pathlib import Path
import random
import warnings
warnings.filterwarnings('ignore')

# Load prepared training data
def load_training_data():

    with open('spacy_training_data.json', 'r') as f:
        data = json.load(f)

    # Convert back to spaCy format
    training_data = []
    for item in data:
        text = item['text']
        entities = item['entities']
        training_data.append((text, {"entities": entities}))

    return training_data

# Train transformer-based NER model
def train_transformer_ner(training_data, n_iter=50, batch_size=8, dropout=0.1, model_name="en_core_web_sm"):

    print("="*60)
    print("TRAINING TRANSFORMER NER MODEL")
    print("="*60)

    # Initialize model
    print(f"\n1. Initializing model: {model_name}")
    nlp = spacy.load(model_name)

    # Remove existing NER component if present
    if "ner" in nlp.pipe_names:
        nlp.remove_pipe("ner")

    # Add new NER component
    ner = nlp.add_pipe("ner", last=True)

    # Add labels
    print("\n2. Adding entity labels...")
    labels = set()
    for _, annotations in training_data:
        for ent in annotations.get("entities", []):
            labels.add(ent[2])
            ner.add_label(ent[2])

    print(f"Labels: {', '.join(sorted(labels))}")

    # Convert to Example objects
    print("\n3. Preparing training examples...")
    examples = []
    for text, annots in training_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annots)
        examples.append(example)

    print(f"Total examples: {len(examples)}")

    # Initialize the model
    print("\n4. Initializing training...")
    nlp.initialize(lambda: examples)

    # Training
    print(f"\n5. Training model...")
    print(f"  -  Iterations: {n_iter}")
    print(f"  -  Batch size: {batch_size}")
    print(f"  -  Dropout: {dropout}")

    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

    best_loss = float('inf')
    patience = 5
    no_improvement_count = 0

    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.resume_training()

        for iteration in range(n_iter):
            random.shuffle(examples)
            losses = {}

            # Create minibatches with compounding size
            batches = minibatch(examples, size=compounding(4.0, batch_size, 1.001))

            for batch in batches:
                nlp.update(
                    batch,
                    drop=dropout,
                    losses=losses,
                    sgd=optimizer
                )

            current_loss = losses.get('ner', 0)

            # Print progress
            if iteration % 5 == 0:
                print(f" Iteration {iteration:3d}/{n_iter}: Loss = {current_loss:.4f}")

            # Early stopping check
            if current_loss < best_loss:
                best_loss = current_loss
                no_improvement_count = 0
            else:
                no_improvement_count += 1

            if no_improvement_count >= patience and iteration > 20:
                print(f"\n   Early stopping at iteration {iteration} (no improvement for {patience} iterations)")
                break

            # Stop if loss is very low
            if current_loss < 0.001:
                print(f"\n   Stopping at iteration {iteration} (loss < 0.001)")
                break

    print(f"\n   Final loss: {current_loss:.4f}")

    return nlp

#  Evaluate trained model on test data
def evaluate_model(nlp, test_file='test_data.csv'):


    print("\n" + "="*60)
    print("EVALUATING MODEL ON TEST DATA")
    print("="*60)

    # Load test data
    test_df = pd.read_csv(test_file)
    print(f"\nEvaluating on {len(test_df)} test examples...")

    results = []
    correct_predictions = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}
    total_predictions = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}

    for idx, row in test_df.iterrows():
        if pd.isna(row.get('text')):
            continue

        text = str(row['text'])
        doc = nlp(text)

        # Extract entities
        extracted = {
            'event': '',
            'venue': '',
            'datetime': ''
        }

        for ent in doc.ents:
            if ent.label_ == 'EVENT' and not extracted['event']:
                extracted['event'] = ent.text
            elif ent.label_ == 'VENUE' and not extracted['venue']:
                extracted['venue'] = ent.text
            elif ent.label_ == 'DATETIME' and not extracted['datetime']:
                extracted['datetime'] = ent.text

        # Compare with ground truth
        ground_truth = {
            'event': str(row.get('event', '')) if pd.notna(row.get('event')) else '',
            'venue': str(row.get('venue', '')) if pd.notna(row.get('venue')) else '',
            'datetime': str(row.get('datetime', '')) if pd.notna(row.get('datetime')) else ''
        }

        # Count correct predictions (partial match)
        if ground_truth['event'] and extracted['event']:
            total_predictions['EVENT'] += 1
            if extracted['event'].lower() in ground_truth['event'].lower() or \
               ground_truth['event'].lower() in extracted['event'].lower():
                correct_predictions['EVENT'] += 1

        if ground_truth['venue'] and extracted['venue']:
            total_predictions['VENUE'] += 1
            if extracted['venue'].lower() in ground_truth['venue'].lower() or \
               ground_truth['venue'].lower() in extracted['venue'].lower():
                correct_predictions['VENUE'] += 1

        if extracted['datetime']:
            total_predictions['DATETIME'] += 1
            if ground_truth['datetime']:
                correct_predictions['DATETIME'] += 1

        results.append({
            'text': text[:100],
            'extracted_event': extracted['event'],
            'extracted_venue': extracted['venue'],
            'extracted_datetime': extracted['datetime'],
            'ground_truth_event': ground_truth['event'],
            'ground_truth_venue': ground_truth['venue'],
            'ground_truth_datetime': ground_truth['datetime']
        })

    # Calculate accuracy
    print("\n Accuracy by entity type:")
    for entity_type in ['EVENT', 'VENUE', 'DATETIME']:
        if total_predictions[entity_type] > 0:
            accuracy = correct_predictions[entity_type] / total_predictions[entity_type] * 100
            print(f"   {entity_type}: {correct_predictions[entity_type]}/{total_predictions[entity_type]} = {accuracy:.1f}%")
        else:
            print(f"   {entity_type}: No predictions made")

    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv('model_evaluation_results.csv', index=False)
    print(f"\n Detailed results saved to: model_evaluation_results.csv")

    # Show examples
    print("\n Sample predictions:")
    print("-"*60)
    for i in range(min(3, len(results))):
        result = results[i]
        print(f"\n Example {i+1}:")
        print(f"Text: {result['text']}...")
        print(f"Extracted Event: '{result['extracted_event']}'")
        print(f"Ground Truth Event: '{result['ground_truth_event']}'")
        print(f"Extracted Venue: '{result['extracted_venue']}'")
        print(f"Ground Truth Venue: '{result['ground_truth_venue']}'")

    return results

# Save trained model
def save_model(nlp, output_dir="transformer_ner_model"):

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    nlp.to_disk(output_path)
    print(f"\n✅ Model saved to: {output_dir}/")
    return output_path

# Main training pipeline
def main():

    print("TRANSFORMER NER TRAINING PIPELINE")
    print("="*60)

    # Load training data
    print("\nStep 1: Loading training data...")
    training_data = load_training_data()
    print(f"   Loaded {len(training_data)} training examples")

    # Train model
    print("\nStep 2: Training model...")
    nlp = train_transformer_ner(
        training_data,
        n_iter=50,
        batch_size=8,
        dropout=0.1,
        model_name="en_core_web_sm"  # Change to "en_core_web_trf" for transformer
    )

    # Save model
    print("\nStep 3: Saving model...")
    save_model(nlp)

    # Evaluate model
    print("\nStep 4: Evaluating model...")
    results = evaluate_model(nlp)

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)


if __name__ == "__main__":
    main()

TRANSFORMER NER TRAINING PIPELINE

Step 1: Loading training data...
   Loaded 418 training examples

Step 2: Training model...
TRAINING TRANSFORMER NER MODEL

1. Initializing model: en_core_web_sm

2. Adding entity labels...
Labels: DATETIME, EVENT, VENUE

3. Preparing training examples...
Total examples: 418

4. Initializing training...

5. Training model...
  -  Iterations: 50
  -  Batch size: 8
  -  Dropout: 0.1
 Iteration   0/50: Loss = 5462.9487
 Iteration   5/50: Loss = 446.8034
 Iteration  10/50: Loss = 190.3278
 Iteration  15/50: Loss = 107.4400
 Iteration  20/50: Loss = 111.4193
 Iteration  25/50: Loss = 127.5484
 Iteration  30/50: Loss = 56.3540
 Iteration  35/50: Loss = 101.8529

   Early stopping at iteration 38 (no improvement for 5 iterations)

   Final loss: 62.5155

Step 3: Saving model...

✅ Model saved to: transformer_ner_model/

Step 4: Evaluating model...

EVALUATING MODEL ON TEST DATA

Evaluating on 76 test examples...

 Accuracy by entity type:
   EVENT: 1/7 = 14.

In [ ]:
no

## **Step 5:** Debug and Fix Transformer NER Model -- Better Model with other parameters (learning_rate, dropout, increased iterations, and batch_size)

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Diagnose issues with training data
def diagnose_training_data():

    print("="*60)
    print("Diagnosing Training Data")
    print("="*60)

    # Load training data
    with open('spacy_training_data.json', 'r') as f:
        data = json.load(f)

    print(f"\n1. Total training examples: {len(data)}")

    # Check entity distribution (venue, datyatime, event)
    entity_counts = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}
    empty_examples = 0

    for item in data:
        if not item['entities']:
            empty_examples += 1
            continue

        for start, end, label in item['entities']:
            entity_counts[label] = entity_counts.get(label, 0) + 1

    print(f"\n2. Entity distribution:")
    for label, count in entity_counts.items():
        print(f"   {label}: {count}")

    print(f"\n3. Empty examples: {empty_examples}")

    # Check a few examples
    print(f"\n4. Sample training examples:")
    for i in range(min(3, len(data))):
        text = data[i]['text'][:100]
        entities = data[i]['entities']
        print(f"\nExample {i+1}:")
        print(f"Text: {text}...")
        if entities:
           if entities:
            for start, end, label in entities[:2]:  # Show first 2 entities
                entity_text = data[i]['text'][start:end]
                print(f"  {label}: '{entity_text}' [{start}:{end}]")
        else:
            print("  No entities")

    return data

# Train with better configuration and more iterations
def train_with_better_config():


    print("\n" + "="*60)
    print("Retraining w/ Better Configuration")
    print("="*60)

    # Load training data
    with open('spacy_training_data.json', 'r') as f:
        json_data = json.load(f)

    # Convert to spaCy format
    training_data = []
    for item in json_data:
        training_data.append((item['text'], {"entities": item['entities']}))

    print(f"\n1. Loaded {len(training_data)} training examples")

    # Use a better base model if available
    try:
        # Try to use the large model (ref. en_core_web_lg)
        nlp = spacy.load("en_core_web_lg")
        print("2. Using en_core_web_lg (large model)")
    except:
        try:
            # Fall back to medium
            nlp = spacy.load("en_core_web_md")
            print("2. Using en_core_web_md (medium model)")
        except:
            # Use small model
            nlp = spacy.load("en_core_web_sm")
            print("2. Using en_core_web_sm (small model)")

    # Remove old NER and add new one
    if "ner" in nlp.pipe_names:
        nlp.remove_pipe("ner")

    # Add NER with config
    ner = nlp.add_pipe("ner", last=True)

    # Add labels
    for _, annotations in training_data:
        for ent in annotations.get("entities", []):
            ner.add_label(ent[2])

    print(f"3. Added labels: {ner.labels}")

    # Prepare examples
    examples = []
    for text, annots in training_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annots)
        examples.append(example)

    # Initialize
    nlp.initialize(lambda: examples)

    # Training with better parameters (patience, dropout, iterations, batch size, learning rate)
    print("\n4. Training with improved parameters...")
    print("   - Iterations: 100 (increased)")
    print("   - Batch size: 16 (increased)")
    print("   - Dropout: 0.2 (increased)")
    print("   - Learning rate: Adaptive")

    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

    best_loss = float('inf')
    patience_counter = 0
    patience = 10  # Patienc increasede

    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.resume_training()
        #More iterations
        for iteration in range(100):
            random.shuffle(examples)
            losses = {}

            # Use larger batches
            batches = minibatch(examples, size=compounding(8.0, 16.0, 1.001))

            for batch in batches:
                nlp.update(
                    batch,
                    drop=0.2,  # Higher dropout
                    losses=losses,
                    sgd=optimizer
                )

            current_loss = losses.get('ner', 0)

            # Print progress
            if iteration % 10 == 0:
                print(f"   Iteration {iteration:3d}: Loss = {current_loss:.4f}")

            # Better early stopping
            if current_loss < best_loss * 0.99:  # Only improved if >1% better
                best_loss = current_loss
                patience_counter = 0
                # Save best model
                nlp.to_disk("transformer_ner_model_best")
            else:
                patience_counter += 1

            if patience_counter >= patience and iteration > 30:
                print(f"\n   Early stopping at iteration {iteration}")

                # Load best model
                nlp = spacy.load("transformer_ner_model_best")
                break

    print(f"\n   Final loss: {current_loss:.4f}")

    # Save final model
    nlp.to_disk("transformer_ner_model_improved")
    print("\n5. Model saved to: transformer_ner_model_improved/")

    return nlp

# Test the improved model
def test_improved_model(nlp):

    print("\n" + "="*60)
    print("Testing Improved Model")
    print("="*60)

    test_texts = [
        "Concert at Red Rocks on Friday at 7pm",
        "Denver Jazz Festival this weekend at City Park",
        "Comedy show tonight at Comedy Works",
        "I Love R&B Party at Your Mom's House Denver",
        "Mixtape Saturdays Live Music @ Punch Bowl Social"
    ]

    print("\nTest Results:")
    for text in test_texts:
        doc = nlp(text)
        print(f"\nText: {text}")

        entities = [(ent.text, ent.label_) for ent in doc.ents]

        if entities:
            for ent_text, label in entities:
                print(f"  {label}: '{ent_text}'")
        else:
            print("  ❌ No entities found")

    # Test on actual data
    print("\n" + "-"*60)
    print("Testing on actual test data:")

    test_df = pd.read_csv('test_data.csv')

    successes = 0
    total = 0

    for _, row in test_df.head(10).iterrows():
        if pd.isna(row.get('text')):
            continue

        doc = nlp(row['text'])

        extracted = {}
        for ent in doc.ents:
            if ent.label_ not in extracted:
                extracted[ent.label_] = ent.text

        # Check if we got anything
        if row.get('event') and not pd.isna(row.get('event')):
            total += 1
            if extracted.get('EVENT'):
                successes += 1

        if row.get('venue') and not pd.isna(row.get('venue')):
            total += 1
            if extracted.get('VENUE'):
                successes += 1

    if total > 0:
        print(f"\nQuick accuracy: {successes}/{total} = {successes/total*100:.1f}%")

    return nlp

# Add rule-based patterns to help the model
def add_pattern_matcher(nlp):

    from spacy.matcher import Matcher

    print("\n" + "="*60)
    print("ADDING PATTERN-BASED RULES")
    print("="*60)

    # Create rule_based pattern matcher in SpaCY
    # Use vacab (word knowledge, nlp.vacab from SpaCY model
    matcher = Matcher(nlp.vocab)

    # Event patterns
    event_patterns = [
        [{"LOWER": {"IN": ["concert", "festival", "show", "party", "event"]}},
         {"IS_ALPHA": True, "OP": "*"}],
        [{"IS_TITLE": True, "OP": "+"},
         {"LOWER": {"IN": ["festival", "concert", "show", "party"]}}],
    ]

    # Venue patterns (after "at" or "@")
    venue_patterns = [
        [{"LOWER": {"IN": ["at", "@"]}},
         {"IS_TITLE": True, "OP": "+"}],
    ]

    for i, pattern in enumerate(event_patterns):
        matcher.add(f"EVENT_PATTERN_{i}", [pattern])

    for i, pattern in enumerate(venue_patterns):
        matcher.add(f"VENUE_PATTERN_{i}", [pattern])

    print("Added pattern rules for EVENT and VENUE detection")

    return nlp, matcher

# Main retraining pipeline
def main():

    print("IMPROVED NER TRAINING PIPELINE")
    print("="*60)

    # Step 1: Diagnose data
    training_data = diagnose_training_data()

    # Step 2: Retrain with better config
    nlp = train_with_better_config()

    # Step 3: Test improved model
    test_improved_model(nlp)

    # Step 4: Add patterns (optional)
    nlp, matcher = add_pattern_matcher(nlp)

    print("\n" + "="*60)
    print("✅ IMPROVED TRAINING COMPLETE")
    print("="*60)
    print("\nUse: nlp = spacy.load('transformer_ner_model_improved')")

if __name__ == "__main__":
    main()

IMPROVED NER TRAINING PIPELINE
Diagnosing Training Data

1. Total training examples: 418

2. Entity distribution:
   EVENT: 349
   VENUE: 184
   DATETIME: 144

3. Empty examples: 0

4. Sample training examples:

Example 1:
Text: Denver Symphony Pops Concert - Movie Music Spectacular. Orchestra performing popular film soundtrack...
  EVENT: 'Denver Symphony Pops Concert' [0:28]

Example 2:
Text: Austin Country Music Festival at Leander- Cedar Park, April 6, 2024 - Saturday, April 6, 2024 at The...
  EVENT: 'Austin Country Music Festival' [0:29]
  DATETIME: 'April 6, 2024' [54:67]

Example 3:
Text: Action Navigation

Sunday, December 1, 2024 - 6:00 PM to 7:00 PM Zilker Holiday Tree Website Event S...
  DATETIME: 'Sunday, December 1' [19:37]
  EVENT: 'Zilker Holiday Tree' [65:84]

Retraining w/ Better Configuration

1. Loaded 418 training examples
2. Using en_core_web_sm (small model)
3. Added labels: ('DATETIME', 'EVENT', 'VENUE')

4. Training with improved parameters...
   - Iterations:

# **Post-processing the dataset for better performance**

In [ ]:
"""
FINAL POST-PROCESSING SCRIPT
All functions in one file due import errors when multiple steps!
"""

import spacy
import pandas as pd
import re
from typing import Dict
import os

# ============================================================
# PART 1: CLEANING FUNCTIONS
# ============================================================

# Clean up over-extracted event names
def clean_event_name(event_text: str) -> str:

    cleaned = event_text

    # Remove venue/date indicators from end of event name
    venue_indicators = [
        r'\s+at\s+.*$',  # Remove "at [venue]" from end
        r'\s+@\s+.*$',    # Remove "@ [venue]" from end
        r'\s+in\s+.*$',   # Remove "in [venue]" from end
        r'\s+on\s+.*$',   # Remove "on [date]" from end
    ]

    for pattern in venue_indicators:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)

    # Remove trailing punctuation
    cleaned = cleaned.rstrip('.,;:!')

    # Remove common non-event phrases
    non_event_phrases = [
        r'^(Check out|Visit|See|Join us for)\s+',
        r'\s+(tickets|admission|entry|RSVP).*$',
    ]

    for pattern in non_event_phrases:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)

    return cleaned.strip()

#Extract venue using patterns when not found by NER
def extract_venue_from_text(text: str, existing_venue: str = '') -> str:


    if existing_venue:
        return existing_venue

    venue_patterns = [
        r'(?:at|@)\s+([\w\s]+?)(?:\s+on|\s+at\s+\d|$)',
        r'(?:at|@)\s+((?:[A-Z]\w+\s*)+)',
    ]

    # Try pattern matching for venues
    for pattern in venue_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            venue = match.group(1).strip()
            # Clean up the venue
            venue = venue.rstrip('.,;:!')
            if len(venue) > 2 and len(venue) < 50:  # Reasonable venue name length
                return venue

    return ''

# Extract datetime using patterns when not found by NER
def extract_datetime_from_text(text: str, existing_datetime: str = '') -> str:

    if existing_datetime:
        return existing_datetime

    date_patterns = [
        r'\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{4})?',
        r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?',
        r'\b\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)\b',
        r'\b(?:Tonight|Tomorrow|Today)\b',
    ]

    # Try pattern matching for dates/times
    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0)

    return ''

# ============================================================
# PART 2: MAIN PROCESSING FUNCTION
# ============================================================

#Process text with NER and post-processing
def process_text_with_postprocessing(text: str, nlp=None) -> Dict[str, str]:

    # Load model when not provided
    if nlp is None:
        # Find the best available model
        model_paths = [
            'transformer_ner_model_improved',
            'transformer_ner_model_best',
            'transformer_ner_model'
        ]

        model_loaded = False
        for path in model_paths:
            if os.path.exists(path):
                try:
                    nlp = spacy.load(path)
                    print(f"Loaded model from: {path}")
                    model_loaded = True
                    break
                except:
                    continue

        if not model_loaded:
            print("Warning: No trained model found. Using base model.")
            nlp = spacy.load("en_core_web_sm")

    # Get NER predictions
    doc = nlp(text)

    # Initialize extractions
    extracted = {
        'EVENT': '',
        'VENUE': '',
        'DATETIME': ''
    }

    # Collect NER entities
    for ent in doc.ents:
        if ent.label_ == 'EVENT' and not extracted['EVENT']:
            extracted['EVENT'] = clean_event_name(ent.text)
        elif ent.label_ == 'VENUE' and not extracted['VENUE']:
            extracted['VENUE'] = ent.text
        elif ent.label_ == 'DATETIME' and not extracted['DATETIME']:
            extracted['DATETIME'] = ent.text

    # Apply pattern-based extraction for missing fields
    if not extracted['VENUE']:
        extracted['VENUE'] = extract_venue_from_text(text)

    if not extracted['DATETIME']:
        extracted['DATETIME'] = extract_datetime_from_text(text)

    # When event is too long, it might include venue - try to split
    if extracted['EVENT'] and len(extracted['EVENT']) > 50:
        cleaned_event = clean_event_name(extracted['EVENT'])
        if cleaned_event != extracted['EVENT']:
            extracted['EVENT'] = cleaned_event
            # Try to extract venue from the removed part
            if not extracted['VENUE']:
                extracted['VENUE'] = extract_venue_from_text(text)

    return extracted

# ============================================================
# PART 3: TESTING FUNCTIONS
# ============================================================

# Test extraction on sample texts
def test_basic_extraction():

    print("="*70)
    print("TESTING BASIC EXTRACTION")
    print("="*70)

    test_cases = [
        "Concert at Red Rocks on Friday at 7pm",
        "Denver Jazz Festival this weekend at City Park",
        "I Love R&B Party at Your Mom's House Denver",
        "Mixtape Saturdays Live Music @ Punch Bowl Social",
        "Comedy show tonight at Comedy Works Downtown",
        "Beatles Tribute Band at Ball Arena December 15th 8pm",
    ]

    # Load model once
    nlp = None

    print("\nProcessing test cases:")
    print("-"*60)

    for text in test_cases:
        result = process_text_with_postprocessing(text, nlp)

        print(f"\nText: {text}")
        print(f"  EVENT:    '{result['EVENT']}'")
        print(f"  VENUE:    '{result['VENUE']}'")
        print(f"  DATETIME: '{result['DATETIME']}'")

    return True

# Test actual test data
def test_on_real_data():

    print("\n" + "="*70)
    print("TESTING ON REAL DATA")
    print("="*70)

    # Check if test data exists
    if not os.path.exists('test_data.csv'):
        print("❌ Error: test_data.csv not found!")
        print("Please run the train/test split script first.")
        return None

    # Load test data
    test_df = pd.read_csv('test_data.csv')
    print(f"\nLoaded {len(test_df)} test examples")

    # Track metrics
    correct = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}
    total = {'EVENT': 0, 'VENUE': 0, 'DATETIME': 0}

    # Load model once
    nlp = None

    # Process first 10 examples
    print("\nProcessing first 10 examples:")
    print("-"*60)

    for idx, row in test_df.head(10).iterrows():
        if pd.isna(row.get('text')):
            continue

        text = str(row['text'])
        result = process_text_with_postprocessing(text, nlp)

        print(f"\nExample {idx + 1}:")
        print(f"Text: {text[:80]}...")

        # Check EVENT
        if row.get('event') and not pd.isna(row.get('event')) and str(row['event']) != 'nan':
            total['EVENT'] += 1
            ground_truth_event = str(row['event']).strip()

            if result['EVENT']:
                # Fuzzy match
                if (result['EVENT'].lower() in ground_truth_event.lower() or
                    ground_truth_event.lower() in result['EVENT'].lower()):
                    correct['EVENT'] += 1
                    print(f"  ✅ EVENT: '{result['EVENT']}'")
                else:
                    print(f"  ❌ EVENT: '{result['EVENT']}' (expected: '{ground_truth_event}')")
            else:
                print(f"  ❌ EVENT: Not found (expected: '{ground_truth_event}')")

        # Check VENUE
        if row.get('venue') and not pd.isna(row.get('venue')) and str(row['venue']) != 'nan':
            total['VENUE'] += 1
            ground_truth_venue = str(row['venue']).strip()

            if result['VENUE']:
                if (result['VENUE'].lower() in ground_truth_venue.lower() or
                    ground_truth_venue.lower() in result['VENUE'].lower()):
                    correct['VENUE'] += 1
                    print(f"  ✅ VENUE: '{result['VENUE']}'")
                else:
                    print(f"  ❌ VENUE: '{result['VENUE']}' (expected: '{ground_truth_venue}')")
            else:
                print(f"  ❌ VENUE: Not found (expected: '{ground_truth_venue}')")

    # Calculate accuracy
    print("\n" + "="*70)
    print("ACCURACY METRICS")
    print("-"*60)

    for entity_type in ['EVENT', 'VENUE', 'DATETIME']:
        if total[entity_type] > 0:
            accuracy = correct[entity_type] / total[entity_type] * 100
            print(f"{entity_type}: {correct[entity_type]}/{total[entity_type]} = {accuracy:.1f}%")

    overall_correct = sum(correct.values())
    overall_total = sum(total.values())

    if overall_total > 0:
        overall_accuracy = overall_correct / overall_total * 100
        print(f"\nOverall: {overall_correct}/{overall_total} = {overall_accuracy:.1f}%")

        if overall_accuracy > 70:
            print("✅ Good performance!")
        elif overall_accuracy > 60:
            print("⚠️ Moderate performance")
        else:
            print("❌ Needs improvement")

    return correct, total

# Evaluate on entire test set
def evaluate_full_dataset():

    print("\n" + "="*70)
    print("FULL DATASET EVALUATION")
    print("="*70)

    # Check if test data exists
    if not os.path.exists('test_data.csv'):
        print("❌ Error: test_data.csv not found!")
        return None

    # Load test data
    test_df = pd.read_csv('test_data.csv')
    print(f"\nEvaluating all {len(test_df)} test examples...")

    # Initialize metrics
    metrics = {
        'EVENT': {'correct': 0, 'total': 0},
        'VENUE': {'correct': 0, 'total': 0},
        'DATETIME': {'correct': 0, 'total': 0}
    }

    # Store results
    all_results = []

    # Load model for efficiency
    nlp = None

    # Process all examples
    for idx, row in test_df.iterrows():
        if pd.isna(row.get('text')):
            continue

        text = str(row['text'])
        extracted = process_text_with_postprocessing(text, nlp)

        # Save result
        result_row = {
            'text': text[:100],
            'extracted_event': extracted['EVENT'],
            'extracted_venue': extracted['VENUE'],
            'extracted_datetime': extracted['DATETIME'],
            'ground_truth_event': '',
            'ground_truth_venue': '',
            'ground_truth_datetime': ''
        }

        # Check EVENT
        if row.get('event') and not pd.isna(row.get('event')) and str(row['event']) != 'nan':
            ground_truth = str(row['event']).strip()
            result_row['ground_truth_event'] = ground_truth
            metrics['EVENT']['total'] += 1

            if extracted['EVENT']:
                if (extracted['EVENT'].lower() in ground_truth.lower() or
                    ground_truth.lower() in extracted['EVENT'].lower()):
                    metrics['EVENT']['correct'] += 1

        # Check VENUE
        if row.get('venue') and not pd.isna(row.get('venue')) and str(row['venue']) != 'nan':
            ground_truth = str(row['venue']).strip()
            result_row['ground_truth_venue'] = ground_truth
            metrics['VENUE']['total'] += 1

            if extracted['VENUE']:
                if (extracted['VENUE'].lower() in ground_truth.lower() or
                    ground_truth.lower() in extracted['VENUE'].lower()):
                    metrics['VENUE']['correct'] += 1

        # Check DATETIME
        if row.get('datetime') and not pd.isna(row.get('datetime')) and str(row['datetime']) != 'nan':
            ground_truth = str(row['datetime']).strip()
            result_row['ground_truth_datetime'] = ground_truth
            metrics['DATETIME']['total'] += 1

            if extracted['DATETIME']:
                metrics['DATETIME']['correct'] += 1

        all_results.append(result_row)

        # Progress update
        if (idx + 1) % 20 == 0:
            print(f"  Processed {idx + 1}/{len(test_df)}...")

    # Calculate final metrics
    print("\n" + "="*70)
    print("FINAL PERFORMANCE METRICS")
    print("-"*60)

    overall_correct = 0
    overall_total = 0

    for entity_type in ['EVENT', 'VENUE', 'DATETIME']:
        m = metrics[entity_type]
        if m['total'] > 0:
            accuracy = m['correct'] / m['total'] * 100
            print(f"\n{entity_type}:")
            print(f"  Accuracy: {m['correct']}/{m['total']} = {accuracy:.1f}%")

            overall_correct += m['correct']
            overall_total += m['total']

    if overall_total > 0:
        overall_accuracy = overall_correct / overall_total * 100
        print(f"\n" + "-"*60)
        print(f"OVERALL ACCURACY: {overall_correct}/{overall_total} = {overall_accuracy:.1f}%")

        #Interpretation of performance
        print("\nPERFORMANCE LEVEL: ", end="")
        if overall_accuracy > 75:
            print("⭐⭐⭐⭐⭐ EXCELLENT - Production Ready!")
        elif overall_accuracy > 70:
            print("⭐⭐⭐⭐ GOOD - Ready with minor improvements")
        elif overall_accuracy > 65:
            print("⭐⭐⭐ MODERATE - Usable with monitoring")
        elif overall_accuracy > 60:
            print("⭐⭐ FAIR - Needs improvement")
        else:
            print("⭐ POOR - Requires significant work")

    # Save results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('postprocessed_results.csv', index=False)
    print(f"\n✅ Results saved to: postprocessed_results.csv")

    return metrics

# ============================================================
# MAIN EXECUTION
# ============================================================

# Run all tests
def main():

    print("POST-PROCESSING PIPELINE")
    print("="*70)

    # Step 1: Test basic extraction
    print("\nStep 1: Testing basic extraction...")
    test_basic_extraction()

    # Step 2: Test on real data (first 10)
    print("\nStep 2: Testing on real data sample...")
    test_on_real_data()

    # Step 3: Full evaluation
    print("\nStep 3: Running full evaluation...")
    evaluate_full_dataset()

    print("\n" + "="*70)
    print("✅ POST-PROCESSING COMPLETE")
    print("="*70)

if __name__ == "__main__":
    main()

POST-PROCESSING PIPELINE

Step 1: Testing basic extraction...
TESTING BASIC EXTRACTION

Processing test cases:
------------------------------------------------------------
Loaded model from: transformer_ner_model_improved

Text: Concert at Red Rocks on Friday at 7pm
  EVENT:    'Concert'
  VENUE:    'Red Rocks'
  DATETIME: 'Friday'
Loaded model from: transformer_ner_model_improved

Text: Denver Jazz Festival this weekend at City Park
  EVENT:    'Denver Jazz Festival this weekend'
  VENUE:    'City Park'
  DATETIME: ''
Loaded model from: transformer_ner_model_improved

Text: I Love R&B Party at Your Mom's House Denver
  EVENT:    ''
  VENUE:    'Your Mom'
  DATETIME: ''
Loaded model from: transformer_ner_model_improved

Text: Mixtape Saturdays Live Music @ Punch Bowl Social
  EVENT:    'Mixtape Saturdays Live Music'
  VENUE:    'Punch Bowl Social'
  DATETIME: ''
Loaded model from: transformer_ner_model_improved

Text: Comedy show tonight at Comedy Works Downtown
  EVENT:    'Comedy sho

In [ ]:
"""
OPTIMIZED POST-PROCESSING PIPELINE
Fixes model loading issue and improves extraction patterns
"""

import spacy
import pandas as pd
import re
from typing import Dict, Optional
import os

# Optimized NER processor with single model load
class OptimizedNERProcessor:

    # Initialize and load model
    def __init__(self):

        self.nlp = self.load_model()
        self.setup_patterns()

    # Load the best available model
    def load_model(self):

        model_paths = [
            'transformer_ner_model_improved',
            'transformer_ner_model_best',
            'transformer_ner_model'
        ]

        for path in model_paths:
            if os.path.exists(path):
                try:
                    print(f"Loading model from: {path}")
                    return spacy.load(path)
                except:
                    continue

        print("Warning: No trained model found. Using base model.")
        return spacy.load("en_core_web_sm")

    # Setup improved regex patterns
    def setup_patterns(self):

        # Enhanced venue patterns
        self.venue_patterns = [
            r'(?:at|@)\s+([\w\s\']+?)(?:\s+\(|$|\s+on\s+|\s+at\s+\d|\.)',
            r'(?:at|@)\s+((?:[A-Z][\w\']*\s*)+)',
            r'(?:venue:|location:)\s*([\w\s\']+?)(?:\.|,|$)',
        ]

        # Enhanced date patterns
        self.date_patterns = [
            r'\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b',
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{4})?',
            r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?',
            r'\b\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)\b',
            r'\b(?:Tonight|Tomorrow|Today)\b',
        ]

        # Event keywords for fallback
        self.event_keywords = [
            'party', 'concert', 'festival', 'show', 'performance',
            'game', 'match', 'ceremony', 'gala', 'celebration',
            'workshop', 'class', 'seminar', 'conference', 'meetup'
        ]

    # Clean up over-extracted event names
    def clean_event_name(self, event_text: str) -> str:

        cleaned = event_text

        # Remove venue/date indicators
        indicators = [
            r'\s+at\s+.*$',
            r'\s+@\s+.*$',
            r'\s+in\s+.*$',
            r'\s+on\s+.*$',
            r'\.\s+.*$',  # Remove everything after period
        ]

        for pattern in indicators:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)

        # Remove trailing punctuation
        cleaned = cleaned.rstrip('.,;:!')

        return cleaned.strip()

    # Extract venue with improved patterns
    def extract_venue(self, text: str, existing_venue: str = '') -> str:


        if existing_venue and len(existing_venue) > 3:
            # Check if venue name seems incomplete
            if existing_venue.lower() in ['your mom', 'the', 'at']:
                # Try to get full venue name
                for pattern in self.venue_patterns:
                    match = re.search(pattern, text, re.IGNORECASE)
                    if match:
                        venue = match.group(1).strip()
                        if len(venue) > len(existing_venue):
                            return venue.rstrip('.,;:!')
            return existing_venue

        # Try pattern matching
        for pattern in self.venue_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                venue = match.group(1).strip()
                venue = venue.rstrip('.,;:!')
                if 2 < len(venue) < 50:
                    return venue

        return ''

    # Extract datetime with patterns
    def extract_datetime(self, text: str, existing_datetime: str = '') -> str:

        if existing_datetime:
            return existing_datetime

        for pattern in self.date_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return match.group(0)

        return ''

    # Fallback event extraction using keywords
    def extract_event_fallback(self, text: str) -> str:

        text_lower = text.lower()

        # Look for event keywords
        for keyword in self.event_keywords:
            if keyword in text_lower:
                # Try to extract phrase around keyword
                pattern = rf'[\w\s]*{keyword}[\w\s]*'
                match = re.search(pattern, text, re.IGNORECASE)
                if match:
                    event = match.group(0).strip()
                    # Clean it up
                    event = self.clean_event_name(event)
                    if len(event) > 5 and len(event) < 100:
                        return event

        return ''

    # Process text with NER and post-processing
    def process_text(self, text: str) -> Dict[str, str]:
        """Process text with NER and post-processing"""

        # Get NER predictions
        doc = self.nlp(text)

        # Initialize extractions
        extracted = {
            'EVENT': '',
            'VENUE': '',
            'DATETIME': ''
        }

        # Collect NER entities
        for ent in doc.ents:
            if ent.label_ == 'EVENT' and not extracted['EVENT']:
                extracted['EVENT'] = self.clean_event_name(ent.text)
            elif ent.label_ == 'VENUE' and not extracted['VENUE']:
                extracted['VENUE'] = ent.text
            elif ent.label_ == 'DATETIME' and not extracted['DATETIME']:
                extracted['DATETIME'] = ent.text

        # Post-processing improvements
        extracted['VENUE'] = self.extract_venue(text, extracted['VENUE'])
        extracted['DATETIME'] = self.extract_datetime(text, extracted['DATETIME'])

        # Fallback for event if not found
        if not extracted['EVENT']:
            extracted['EVENT'] = self.extract_event_fallback(text)

        # Clean up event if too long
        if extracted['EVENT'] and len(extracted['EVENT']) > 50:
            extracted['EVENT'] = self.clean_event_name(extracted['EVENT'])

        return extracted

# Run optimized evaluation
def run_optimized_evaluation():

    print("="*70)
    print("OPTIMIZED NER EVALUATION")
    print("="*70)

    # Initialize processor once
    processor = OptimizedNERProcessor()
    print("✅ Processor initialized (model loaded once)")

    # Test on sample texts
    print("\n" + "="*70)
    print("TESTING SAMPLE TEXTS")
    print("-"*60)

    test_cases = [
        "I Love R&B Party at Your Mom's House Denver",
        "Mixtape Saturdays Live Music @ Punch Bowl Social",
        "Concert at Red Rocks on Friday at 7pm",
    ]

    for text in test_cases:
        result = processor.process_text(text)
        print(f"\nText: {text}")
        print(f"  EVENT:    '{result['EVENT']}'")
        print(f"  VENUE:    '{result['VENUE']}'")
        print(f"  DATETIME: '{result['DATETIME']}'")

    # Full evaluation
    if os.path.exists('test_data.csv'):
        print("\n" + "="*70)
        print("FULL DATASET EVALUATION")
        print("-"*60)

        test_df = pd.read_csv('test_data.csv')
        print(f"\nEvaluating {len(test_df)} test examples...")

        metrics = {
            'EVENT': {'correct': 0, 'total': 0},
            'VENUE': {'correct': 0, 'total': 0},
            'DATETIME': {'correct': 0, 'total': 0}
        }

        all_results = []

        # Process all examples
        for idx, row in test_df.iterrows():
            if pd.isna(row.get('text')):
                continue

            text = str(row['text'])
            extracted = processor.process_text(text)

            # Check EVENT
            if row.get('event') and not pd.isna(row.get('event')) and str(row['event']) != 'nan':
                ground_truth = str(row['event']).strip()
                metrics['EVENT']['total'] += 1

                if extracted['EVENT']:
                    if (extracted['EVENT'].lower() in ground_truth.lower() or
                        ground_truth.lower() in extracted['EVENT'].lower() or
                        len(set(extracted['EVENT'].lower().split()) & set(ground_truth.lower().split())) >= 2):
                        metrics['EVENT']['correct'] += 1

            # Check VENUE
            if row.get('venue') and not pd.isna(row.get('venue')) and str(row['venue']) != 'nan':
                ground_truth = str(row['venue']).strip()
                metrics['VENUE']['total'] += 1

                if extracted['VENUE']:
                    if (extracted['VENUE'].lower() in ground_truth.lower() or
                        ground_truth.lower() in extracted['VENUE'].lower()):
                        metrics['VENUE']['correct'] += 1

            # Check DATETIME
            if row.get('datetime') and not pd.isna(row.get('datetime')) and str(row['datetime']) != 'nan':
                metrics['DATETIME']['total'] += 1

                if extracted['DATETIME']:
                    metrics['DATETIME']['correct'] += 1

            # Store result
            all_results.append({
                'text': text[:100],
                'extracted_event': extracted['EVENT'],
                'extracted_venue': extracted['VENUE'],
                'extracted_datetime': extracted['DATETIME']
            })

            # Progress
            if (idx + 1) % 25 == 0:
                print(f"  Processed {idx + 1}/{len(test_df)}...")

        # Calculate metrics
        print("\n" + "="*70)
        print("OPTIMIZED PERFORMANCE METRICS")
        print("-"*60)

        overall_correct = 0
        overall_total = 0

        for entity_type in ['EVENT', 'VENUE', 'DATETIME']:
            m = metrics[entity_type]
            if m['total'] > 0:
                accuracy = m['correct'] / m['total'] * 100
                print(f"\n{entity_type}:")
                print(f"  Accuracy: {m['correct']}/{m['total']} = {accuracy:.1f}%")

                overall_correct += m['correct']
                overall_total += m['total']

        if overall_total > 0:
            overall_accuracy = overall_correct / overall_total * 100
            print(f"\n" + "-"*60)
            print(f"OVERALL ACCURACY: {overall_correct}/{overall_total} = {overall_accuracy:.1f}%")

            print("\nPERFORMANCE LEVEL: ", end="")
            if overall_accuracy > 75:
                print("⭐⭐⭐⭐⭐ EXCELLENT")
            elif overall_accuracy > 70:
                print("⭐⭐⭐⭐ GOOD")
            elif overall_accuracy > 65:
                print("⭐⭐⭐ MODERATE")
            elif overall_accuracy > 60:
                print("⭐⭐ FAIR")
            else:
                print("⭐ NEEDS WORK")

        # Save optimized results
        results_df = pd.DataFrame(all_results)
        results_df.to_csv('optimized_results.csv', index=False)
        print(f"\n✅ Results saved to: optimized_results.csv")

if __name__ == "__main__":
    run_optimized_evaluation()
    print("\n" + "="*70)
    print("✅ OPTIMIZED EVALUATION COMPLETE")
    print("="*70)

OPTIMIZED NER EVALUATION
Loading model from: transformer_ner_model_improved
✅ Processor initialized (model loaded once)

TESTING SAMPLE TEXTS
------------------------------------------------------------

Text: I Love R&B Party at Your Mom's House Denver
  EVENT:    'B Party'
  VENUE:    'Your Mom's House Denver'
  DATETIME: ''

Text: Mixtape Saturdays Live Music @ Punch Bowl Social
  EVENT:    'Mixtape Saturdays Live Music'
  VENUE:    'Punch Bowl Social'
  DATETIME: ''

Text: Concert at Red Rocks on Friday at 7pm
  EVENT:    'Concert'
  VENUE:    'Red Rocks'
  DATETIME: 'Friday'

FULL DATASET EVALUATION
------------------------------------------------------------

Evaluating 76 test examples...
  Processed 25/76...
  Processed 50/76...
  Processed 75/76...

OPTIMIZED PERFORMANCE METRICS
------------------------------------------------------------

EVENT:
  Accuracy: 55/76 = 72.4%

VENUE:
  Accuracy: 25/48 = 52.1%

DATETIME:
  Accuracy: 20/24 = 83.3%

----------------------------------

In [ ]:
# Download  optimized_results.csv
from google.colab import files
files.download('optimized_results.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>